# Vector Autoregression of US Growth, Inflation, and the Policy Rate (PROC VARMAX)


## Executive Summary

This notebook models the joint quarterly dynamics of **US real GDP growth**, **CPI inflation**, and the **federal funds rate** with **PROC VARMAX**, fitting a three-variable vector autoregression on six decades of Federal Reserve (FRED) data (1960Q1-2024Q4, 260 quarters). We read the fitted VAR(2) coefficients, trace how a shock to one series propagates through the others (impulse responses), attribute each series' forecast uncertainty to the three innovations (forecast-error variance decomposition), and forecast the next four quarters.

Every number quoted below is taken directly from the executed output. The **fed funds rate is the system's anchor**: its own first lag carries a coefficient of **0.785** (t = 12.11), so policy moves in slow, deliberate steps. The rate also **leans against the economy** - last quarter's growth raises it by **0.300** (t = 4.56), the reduced form of a policy rule. By a six-quarter horizon, **growth innovations explain about 26%** of the fed funds rate's forecast-error variance (up from 4% at one quarter), so much of the policy rate's medium-term uncertainty is really growth uncertainty passing through the reaction function.


## Data Sources

| Dataset | Rows | Frequency | Key variables | Description |
|---|---|---|---|---|
| `macro` (from `data/macro_levels.csv`) | 260 (after differencing) | Quarterly, 1960Q1-2024Q4 | `dlgdp`, `dlcpi`, `ffr` | Real GDP (FRED `GDPC1`) and the consumer price index (`CPIAUCSL`) expressed as quarter-over-quarter percent growth, plus the effective federal funds rate (`FEDFUNDS`) in levels. Pulled once from FRED's public API and committed as a snapshot next to the notebook. |

The committed snapshot keeps the notebook offline and reproducible; the kernel runs uncapped, so all 260 quarters are modeled rather than a truncated preview. To reproduce the pull with live data, use the three-line SASEFRED pattern shown in the [Vector Time Series guide](https://docs.jenneranalytics.com/guides/econometrics/vector-time-series/).


# Vector Autoregression of Growth, Inflation, and the Policy Rate

Macroeconomists and rates strategists watch **growth**, **inflation**, and the **policy rate** as one system. The central bank sets the rate partly in response to growth and inflation; the rate in turn shapes both. A **vector autoregression (VAR)** captures these mutual dynamics in one set of equations, lets us trace how a shock to any series ripples through the others (impulse responses), attributes forecast uncertainty to each source (variance decomposition), and produces coherent joint forecasts.

We fit a **VAR(2)** with **PROC VARMAX** on quarterly FRED data.


## Step 1 - Load the data and build stationary series

Real GDP and the price index trend upward, so we model their quarter-over-quarter percent change rather than their level; the fed funds rate is already a stationary rate and carries through untouched. The first observation has no prior quarter to difference against, so we drop it.


In [1]:
proc import datafile="data/macro_levels.csv" out=levels
    dbms=csv replace;
    getnames=yes;
run;

data macro;
    set levels;
    /* quarterly percent growth of GDP and CPI */
    dlgdp = 100 * (gdp / lag(gdp) - 1);
    dlcpi = 100 * (cpi / lag(cpi) - 1);
    if _n_ > 1;              /* first row has no lag */
    keep dlgdp dlcpi ffr;
run;

proc means data=macro n mean std min max maxdec=3;
    var dlgdp dlcpi ffr;
run;

                                                  The MEANS Procedure

 Variable         N        Mean     Std Dev     Minimum     Maximum
 ------------------------------------------------------------------
 dlgdp          260       0.749       1.062      -7.877       7.762
 dlcpi          260       0.921       0.807      -2.333       3.723
 ffr            260       4.801       3.694       0.050      19.080
 ------------------------------------------------------------------




NOTE: PROC IMPORT datafile=data/macro_levels.csv out=levels

NOTE: Imported 261 rows from data/macro_levels.csv.
NOTE: DATA macro


NOTE: Read 261 rows from levels.
NOTE: Wrote macro (260 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


The three series span **260 quarters**. Growth averages **0.75% per quarter** (about 3% annualized) with a 1.06-point standard deviation, and the sharpest contraction in the sample is **-7.9%** - the 2020 shutdown quarter. Inflation averages **0.92% per quarter**, and the fed funds rate averages **4.8%**, ranging from the near-zero of the 2010s to the **19%** Volcker peak in 1981. All three are stationary - mean-reverting rather than trending - which is what a VAR in levels needs.


## Step 2 - Fit the VAR(2) with impulse responses and variance decomposition

Listing all three series on the `MODEL` statement makes them mutually endogenous: each gets its own equation, and every equation sees two lags of all three series plus a constant. `PRINT=(ESTIMATES IMPULSE(6) DECOMPOSE(6))` adds the impulse-response and variance-decomposition tables, and `OUTEST=` writes the fitted coefficients to a data set we read back in Step 3.


In [2]:
proc varmax data=macro outest="var_fit.csv";
    model dlgdp dlcpi ffr / p=2
          print=(estimates impulse(6) decompose(6));
run;


                                                  The VARMAX Procedure                                                  

Model Type: VAR(2)
Number of Observations: 260
Dependent Variables: dlgdp dlcpi ffr
AR Order (p): 2
MA Order (q): 0

Parameter Estimates
Parameter                Estimate    Std Error      t Value     Pr > |t|
---------                --------    ---------      -------     --------
L1.dlgdp->dlgdp          0.050024     0.064211         0.78       0.4359
L1.dlcpi->dlgdp         -0.296617     0.111319        -2.66       0.0077
L1.ffr->dlgdp           -0.063144     0.063333        -1.00       0.3188
L1.dlgdp->dlcpi          0.041385     0.036814         1.12       0.2609
L1.dlcpi->dlcpi          0.340223     0.063822         5.33       0.0000
L1.ffr->dlcpi            0.113691     0.036311         3.13       0.0017
L1.dlgdp->ffr            0.299875     0.065720         4.56       0.0000
L1.dlcpi->ffr            0.056860     0.113935         0.50       0.6177
L1.ffr->ff


NOTE: PROC VARMAX data=macro

NOTE: Using Python (statsmodels VAR/VARMAX) for VARMAX estimation


**Reading the coefficients.** Each equation regresses one series on two lags of all three plus a constant. The diagonal lag-1 terms give each series' persistence: the fed funds rate is the anchor at **0.785** (t = 12.11) - most of last quarter's rate carries forward - while inflation is moderately sticky at **0.340** and growth is nearly memoryless at **0.050**. The off-diagonals are the cross-dynamics: the rate rises with last quarter's growth (`L1.dlgdp->ffr` = **0.300**, t = 4.56), the reduced form of a policy rule, and higher inflation last quarter predicts *slower* growth (`L1.dlcpi->dlgdp` = **-0.297**, t = -2.66), the familiar drag of an inflation shock. Judge each term against its standard error before building a story on it: `L1.dlcpi->ffr` (0.057, t = 0.50) is indistinguishable from zero.


**Reading the impulse responses.** Step 0 is the identity - a unit shock moves only its own series. Following a shock to the **fed funds rate**, the rate decays slowly (0.785 at one quarter, 0.610 by six), confirming its persistence; growth barely moves (a slight negative); and inflation edges *up* rather than down over the first few quarters - the reduced-form "price puzzle" familiar from macro VARs. Following a shock to **growth**, the fed funds rate rises and stays elevated (0.300, 0.372, 0.379, 0.368 over the first four quarters): the policy rate leans against a hot economy for several quarters, not just one.


**Reading the variance decomposition.** At a one-quarter horizon each series' forecast error is mostly its own innovation. The movement is over time. Growth stays about **95% self-driven** at every horizon. The fed funds rate starts **90.5% self-driven** at one quarter but, by six quarters, **growth innovations explain 26.3%** of its forecast-error variance (up from 3.6%) and inflation another 10.3% - so the policy rate's medium-term uncertainty is increasingly the economy's uncertainty flowing through the reaction function.


## Step 3 - Capture the fitted coefficients with OUTEST=

`OUTEST=` wrote every fitted coefficient to `var_fit.csv` in a tidy long form - one row per (equation, term) with its estimate, standard error, and t statistic - ready to feed a report, a forecast pipeline, or a downstream model. Read it back and print the nine lag-1 terms:


In [3]:
proc import datafile="var_fit.csv" out=coefs
    dbms=csv replace;
    getnames=yes;
run;

proc print data=coefs (obs=9) noobs;
    var equation parameter estimate stderr tvalue;
run;


Equation  Parameter       Estimate        StdErr         tValue
--------  ---------  -------------  ------------  -------------
dlgdp     l1.dlgdp    0.0500242072  0.0642113634   0.7790553653
dlgdp     l1.dlcpi   -0.2966171129  0.1113187662  -2.6645742038
dlgdp     l1.ffr     -0.0631440806  0.0633332354  -0.9970133407
dlcpi     l1.dlgdp    0.0413850559  0.0368143426    1.124155778
dlcpi     l1.dlcpi    0.3402230698  0.0638224604   5.3307733324
dlcpi     l1.ffr      0.1136912679  0.0363108849   3.1310519769
ffr       l1.dlgdp    0.2998751441   0.065720298   4.5628999421
ffr       l1.dlcpi    0.0568595579   0.113934701   0.4990539092
ffr       l1.ffr      0.7852986222  0.0648215344  12.1147798945

... 12 more observations (showing 9 of 21)




NOTE: PROC IMPORT datafile=var_fit.csv out=coefs

NOTE: Imported 21 rows from var_fit.csv.
NOTE: PROC PRINT data=coefs

NOTE: PROC PRINT completed: 9 observations printed, 5 variables


The full data set carries all twenty-one coefficients (two lag matrices plus the three intercepts). Because it is an ordinary data set, the next step in a pipeline can join it, filter it, or forecast from it without re-fitting the model.


## Step 4 - Forecast the next four quarters

`OUTPUT OUT= LEAD=4` extends the system four quarters past the end of the sample, using the estimated dynamics and the final observed state:


In [4]:
proc varmax data=macro;
    model dlgdp dlcpi ffr / p=2;
    output out=fc lead=4;
run;


                                                  The VARMAX Procedure                                                  

Model Type: VAR(2)
Number of Observations: 260
Dependent Variables: dlgdp dlcpi ffr
AR Order (p): 2
MA Order (q): 0

Parameter Estimates
Parameter                Estimate    Std Error      t Value     Pr > |t|
---------                --------    ---------      -------     --------
L1.dlgdp->dlgdp          0.050024     0.064211         0.78       0.4359
L1.dlcpi->dlgdp         -0.296617     0.111319        -2.66       0.0077
L1.ffr->dlgdp           -0.063144     0.063333        -1.00       0.3188
L1.dlgdp->dlcpi          0.041385     0.036814         1.12       0.2609
L1.dlcpi->dlcpi          0.340223     0.063822         5.33       0.0000
L1.ffr->dlcpi            0.113691     0.036311         3.13       0.0017
L1.dlgdp->ffr            0.299875     0.065720         4.56       0.0000
L1.dlcpi->ffr            0.056860     0.113935         0.50       0.6177
L1.ffr->ff


NOTE: PROC VARMAX data=macro

NOTE: Using Python (statsmodels VAR/VARMAX) for VARMAX estimation
NOTE: Output dataset fc has 4 observations


The four-quarter path extends the system from the end of 2024. Growth holds near **0.8% per quarter** (about 3.3% annualized), inflation drifts up from **0.58% toward 0.80%** per quarter, and the fed funds rate eases gently from **4.60% to 4.51%** - the joint path implied by the estimated dynamics and the final observed state, the input a strategist would carry into a rates or duration view.


## Interpreting the results

- **Persistence.** The fed funds rate is the system's anchor (own lag 0.785, t = 12.11); inflation is moderately sticky (0.340); growth is close to quarter-to-quarter noise (0.050).
- **Policy rule.** The rate rises with last quarter's growth (0.300, t = 4.56), and in the variance decomposition growth innovations come to explain a quarter of its forecast uncertainty by six quarters - the reaction function showing up two ways.
- **Inflation drag.** Higher inflation last quarter predicts slower growth (-0.297, t = -2.66).
- **Impulse responses.** Shocks propagate the way the coefficients imply - a growth shock lifts the policy rate for several quarters; a rate shock decays slowly and leaves growth roughly unchanged.
- **Forecast.** The four-quarter path shows steady growth, firming inflation, and a gently easing policy rate.

**Caveats.** This is a compact three-variable VAR(2) in *reduced* form: its cross-effects are predictive, not structural, and the impulse responses are not identified causal effects without further assumptions (a Cholesky ordering or sign restrictions). A production model might add series (a long rate, unemployment), test for structural breaks across the 1960-2024 span, and check residual whiteness before trusting the forecasts. The coefficients here are cross-validated against an independent R `vars::VAR` reference by regression test `71010`.
